In [ ]:
"""
Netflix Recommendation System using SVD
----------------------------------------
This script loads the Netflix dataset, preprocesses it,
filters out users/movies with too few ratings, trains
an SVD collaborative filtering model, and generates
top movie predictions for a sample user.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Mount Google Drive (only if running on Colab)
from google.colab import drive
drive.mount("/content/drive")

# ------------------------------------------------------------------
# 1. LOAD RATINGS DATA
#    The file is assumed to be a text file with one rating per line:
#    MovieID:  (as a header line) followed by <CustId, Rating> pairs.
# ------------------------------------------------------------------
netflix_dataset = pd.read_csv(
    "/content/drive/MyDrive/netflix data/Copy of Copy of combinedNetflixData (2).txt",
    header=None,
    names=['CustId', 'Rating'],
    usecols=(0, 1)
)

print("Dataset shape:", netflix_dataset.shape)
netflix_dataset.head(20)

# ------------------------------------------------------------------
# 2. EXPLORE DATA
#    Check for missing values and count unique customers.
# ------------------------------------------------------------------
movies_count = netflix_dataset.isnull().sum()['Rating']
total_count = netflix_dataset['CustId'].nunique()
customer_count = total_count - movies_count
rating_count = netflix_dataset['CustId'].count() - movies_count

print(f"Missing ratings: {movies_count}")
print(f"Total unique entries: {total_count}")
print(f"Actual customers: {customer_count}")
print(f"Total ratings: {rating_count}")

# ------------------------------------------------------------------
# 3. EXTRACT MOVIE IDs
#    The dataset uses a special format: lines like "123:" indicate a new movie.
#    We propagate the movie ID to all subsequent rating rows.
# ------------------------------------------------------------------
movie_id = None
movie_id_per_row = []

for i in netflix_dataset['CustId']:
    if ':' in i:                     # line with movie ID
        movie_id = int(i.replace(':', ''))
    movie_id_per_row.append(movie_id)

netflix_dataset['Movie_ID'] = movie_id_per_row

# Remove rows with missing ratings (where 'Rating' was NaN)
netflix_dataset.dropna(inplace=True)

# Convert CustId to integer (it was read as string/object)
netflix_dataset['CustId'] = netflix_dataset['CustId'].astype('int')

print("After cleaning:", netflix_dataset.shape)
netflix_dataset.info()

# ------------------------------------------------------------------
# 4. FILTER MOVIES WITH TOO FEW RATINGS
#    Keep only movies that have at least the 60th percentile of rating counts.
# ------------------------------------------------------------------
summary = netflix_dataset.groupby("Movie_ID")['Rating'].agg(['count'])
movie_benchmark = round(np.percentile(summary['count'], 60, method='linear'))
print(f"Movie count threshold (60th percentile): {movie_benchmark}")

drop_movie_list = summary[summary['count'] < movie_benchmark].index
print(f"Dropping {len(drop_movie_list)} movies with too few ratings.")

# ------------------------------------------------------------------
# 5. FILTER USERS WITH TOO FEW RATINGS
#    Apply similar threshold for customers.
# ------------------------------------------------------------------
summary_cust = netflix_dataset.groupby("CustId")['Rating'].agg(['count'])
cust_benchmark = round(np.percentile(summary_cust['count'], 60, method='linear'))
print(f"User count threshold (60th percentile): {cust_benchmark}")

drop_cust_list = summary_cust[summary_cust['count'] < cust_benchmark].index
print(f"Dropping {len(drop_cust_list)} users with too few ratings.")

# Apply the filters
netflix_dataset = netflix_dataset[~netflix_dataset['Movie_ID'].isin(drop_movie_list)]
netflix_dataset = netflix_dataset[~netflix_dataset['CustId'].isin(drop_cust_list)]

print("Final dataset shape after filtering:", netflix_dataset.shape)

# ------------------------------------------------------------------
# 6. LOAD MOVIE TITLES (for interpretation later)
#    The movies.csv file contains movieId, title, and genres.
# ------------------------------------------------------------------
df_title = pd.read_csv(
    '/content/drive/MyDrive/netflix data/movies (1) (1).csv',
    encoding='ISO-8859-1',
    usecols=[0, 1, 2]
)
df_title.rename(columns={'movieId': 'Movie_ID'}, inplace=True)
print("Movie titles loaded:", df_title.shape)

# ------------------------------------------------------------------
# 7. INSTALL REQUIRED PACKAGES (if not already)
#    We need a compatible numpy version and scikit-surprise.
# ------------------------------------------------------------------
!pip install numpy==1.26.4
!pip install scikit-surprise

from surprise import Reader, Dataset, SVD
from surprise.model_selection import cross_validate

# ------------------------------------------------------------------
# 8. PREPARE DATA FOR SURPRISE
#    Use the first 100,000 rows for faster training (you can remove the slice).
#    The Reader defines the rating scale (1–5).
# ------------------------------------------------------------------
reader = Reader()
data = Dataset.load_from_df(
    netflix_dataset[['CustId', 'Movie_ID', 'Rating']][:100000],
    reader
)

# ------------------------------------------------------------------
# 9. BUILD AND CROSS-VALIDATE THE SVD MODEL
#    SVD is a matrix-factorization algorithm that predicts ratings.
# ------------------------------------------------------------------
model = SVD()
cross_validate(model, data, measures=['RMSE'], cv=3, verbose=True)

# ------------------------------------------------------------------
# 10. GENERATE RECOMMENDATIONS FOR A SPECIFIC USER
#     Pick a sample user (CustId = 1331154). Predict ratings for all movies
#     that the user hasn't rated (and that survived the filtering).
# ------------------------------------------------------------------
sample_user = 1331154
user_ratings = netflix_dataset[netflix_dataset['CustId'] == sample_user]

# Prepare a copy of the movie titles, excluding movies that were dropped earlier
user_predictions = df_title.copy()
user_predictions = user_predictions[~user_predictions['Movie_ID'].isin(drop_movie_list)]

# Predict the estimated score for each movie
user_predictions['Estimated_score'] = user_predictions['Movie_ID'].apply(
    lambda x: model.predict(sample_user, x).est
)

# Sort descending and show top recommendations
user_predictions.sort_values('Estimated_score', ascending=False, inplace=True)
print("Top 5 movie recommendations for user", sample_user)
print(user_predictions.head())